# WaferGuard ML — Financial Impact Analysis
### Phase 2: Anomaly Detection → Batch Frequency → Financial Recommendations

This notebook is the direct continuation of `model_testing.ipynb`.  
It takes the trained models and produces a **ranked financial impact report** for a production batch.

**Pipeline:**
```
artifacts.pkl  (models + test data from model.ipynb)
    → Run both models on full batch
    → Count detections per pattern  ← frequency pre-step
    → Join to financial mapping table
    → Compute weighted daily loss, break-even, EVoA, priority score
    → Ranked repair recommendations
```
---

In [2]:
import json
import os
import sys

import numpy as np

print("✓ Imports ready")

✓ Imports ready


In [3]:
# ── Configuration: Batch settings and financial assumptions ──────────────────
BATCH_ID = "LOT_001"  # Production lot identifier
CONFIDENCE_THRESHOLD = 0.70  # Flag predictions below this for manual review
WPH = 100  # Wafers per hour through affected tool
VALUE_PER_WAFER = 5_000  # USD per wafer
REPAIR_HOURS = 8  # Estimated downtime hours per repair event
PLANNING_HORIZON = 30  # Days used for EVoA calculation

print("✓ Configuration set")
print(f"  Batch ID         : {BATCH_ID}")
print(f"  WPH              : {WPH}")
print(f"  Value / wafer    : ${VALUE_PER_WAFER:,}")
print(f"  Conf. threshold  : {CONFIDENCE_THRESHOLD:.0%}")
print(f"  Planning horizon : {PLANNING_HORIZON} days")

✓ Configuration set
  Batch ID         : LOT_001
  WPH              : 100
  Value / wafer    : $5,000
  Conf. threshold  : 70%
  Planning horizon : 30 days


## Production Pipeline: Run Refactored Library
Execute the refactored financial analysis pipeline using the `waferguard.financial` library.  
Set configuration above, then run this cell to generate reports.


In [ ]:
# Run using real models + dataset when available; otherwise fall back to synthetic demo

from financial_impact import financial as wg_fin

repo_root = "/Users/hernanmontoyag/Desktop/Capstone/waferguard-ml"
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)
print("Using repo_root:", repo_root)

# helper: attempt to load artifacts.pkl, else build from dataset and model files
artifact_dir = globals().get("ARTIFACT_DIR", "training_artifacts")
models_dir = os.path.join(repo_root, "wafer_images", "phase2_models")
models_dir = os.path.abspath(models_dir)
print("models_dir=", models_dir)

# Try direct artifacts first
artifacts = None
try:
    artifacts = wg_fin.load_artifacts(artifact_dir)
    print("Loaded artifacts from", artifact_dir)
except Exception as e:
    print("No artifacts.pkl; will try to reconstruct mappings from model_testing and data:", e)

if artifacts is None:
    # Reconstruct label mapping from model_testing.ipynb (hardcode here same mapping)
    label_mapping = {
        "00000000": "Normal",
        "10000000": "Center",
        "01000000": "Donut",
        "00100000": "Edge_Loc",
        "00010000": "Edge_Ring",
        "00001000": "Loc",
        "00000100": "Near_Full",
        "00000010": "Scratch",
        "00000001": "Random",
        "10100000": "Center+Edge_Loc",
        "10010000": "Center+Edge_Ring",
        "10001000": "Center+Loc",
        "10000010": "Center+Scratch",
        "01100000": "Donut+Edge_Loc",
        "01010000": "Donut+Edge_Ring",
        "01001000": "Donut+Loc",
        "01000010": "Donut+Scratch",
        "00101000": "Edge_Loc+Loc",
        "00100010": "Edge_Loc+Scratch",
        "00011000": "Edge_Ring+Loc",
        "00010010": "Edge_Ring+Scratch",
        "00001010": "Loc+Scratch",
        "10101000": "Center+Edge_Loc+Loc",
        "10100010": "Center+Edge_Loc+Scratch",
        "10011000": "Center+Edge_Ring+Loc",
        "10010010": "Center+Edge_Ring+Scratch",
        "10001010": "Center+Loc+Scratch",
        "01101000": "Donut+Edge_Loc+Loc",
        "01100010": "Donut+Edge_Loc+Scratch",
        "01011000": "Donut+Edge_Ring+Loc",
        "01010010": "Donut+Edge_Ring+Scratch",
        "01001010": "Donut+Loc+Scratch",
        "00101010": "Edge_Loc+Loc+Scratch",
        "00011010": "Edge_Ring+Loc+Scratch",
        "10101010": "Center+Edge_Loc+Loc+Scratch",
        "10011010": "Center+Edge_Ring+Loc+Scratch",
        "01101010": "Donut+Edge_Loc+Loc+Scratch",
        "01011010": "Donut+Edge_Ring+Loc+Scratch",
    }
    unique_patterns = sorted(label_mapping.keys())
    pattern_to_id = {pattern: idx for idx, pattern in enumerate(unique_patterns)}
    id_to_pattern = {idx: label_mapping[pattern] for pattern, idx in pattern_to_id.items()}
    id_to_binary = {v: k for k, v in pattern_to_id.items()}  # reverse mapping: class id -> binary string
    # Load dataset
    data_path = os.path.join(repo_root, "data", "mixedtype-wafer-defect-datasets", "Wafer_Map_Datasets.npz")
    print("Loading dataset from", data_path)
    data = np.load(data_path)
    images = data["arr_0"]
    labels = data["arr_1"]
    images[images == 3] = 0
    # Build X_test (one-hot channels)
    X_all = np.eye(3, dtype=np.float32)[images.astype(int)]
    # Build label string array -> class ids
    label_str_arr = np.array(["".join(map(str, map(int, row))) for row in labels])
    y_class_all = np.array([pattern_to_id.get(s, 0) for s in label_str_arr])
    # Use full batch from dataset as demo
    X_batch = X_all
    print("Loaded dataset shape:", X_batch.shape)
    # Load models from wafer_images/phase2_models
    model_cnn_path = os.path.join(models_dir, "model_p2_cnn.keras")
    model_tl_path = os.path.join(models_dir, "model_p2_tl.keras")
    print("Model paths:", model_cnn_path, model_tl_path)
    # Run inference
    try:
        pred_cnn, pred_tl = wg_fin.run_inference(model_cnn_path, model_tl_path, X_batch)
    except Exception as e:
        print("Model inference failed:", e)
        raise
    class_ids, confidences, use_cnn = wg_fin.merge_predictions(pred_cnn, pred_tl)
    # Decode labels using constructed mapping
    binary_labels, pattern_names = wg_fin.decode_labels(class_ids, id_to_binary, id_to_pattern)
    df_batch2 = wg_fin.build_batch_df(binary_labels, confidences, label_mapping)
    BATCH_ID = globals().get("BATCH_ID", "LOT_001")
    WPH = globals().get("WPH", 100)
    VALUE_PER_WAFER = globals().get("VALUE_PER_WAFER", 5000)
    REPAIR_HOURS = globals().get("REPAIR_HOURS", 8)
    PLANNING_HORIZON = globals().get("PLANNING_HORIZON", 30)
    CONFIDENCE_THRESHOLD = globals().get("CONFIDENCE_THRESHOLD", 0.7)
    config = {
        "WPH": WPH,
        "VALUE_PER_WAFER": VALUE_PER_WAFER,
        "REPAIR_HOURS": REPAIR_HOURS,
        "PLANNING_HORIZON": PLANNING_HORIZON,
        "BATCH_ID": BATCH_ID,
        "total_wafers": len(binary_labels),
        "low_conf_count": int((confidences < CONFIDENCE_THRESHOLD).sum()),
    }
    df_financial2, summary2 = wg_fin.compute_financials(df_batch2, config)
else:
    # artifacts present: use them directly
    X_batch = artifacts.get("X_test")
    if X_batch is None:
        print("Artifacts did not contain X_test; aborting real run.")
        raise SystemExit
    model_cnn_path = os.path.join(repo_root, "wafer_images", "phase2_models", "model_p2_cnn.keras")
    model_tl_path = os.path.join(repo_root, "wafer_images", "phase2_models", "model_p2_tl.keras")
    pred_cnn, pred_tl = wg_fin.run_inference(model_cnn_path, model_tl_path, X_batch)
    class_ids, confidences, use_cnn = wg_fin.merge_predictions(pred_cnn, pred_tl)
    id_to_binary = artifacts.get("id_to_binary", {})
    id_to_pattern = artifacts.get("id_to_pattern", {})
    binary_labels, pattern_names = wg_fin.decode_labels(class_ids, id_to_binary, id_to_pattern)
    df_batch2 = wg_fin.build_batch_df(binary_labels, confidences, artifacts.get("label_mapping", {}))
    config = {
        "WPH": globals().get("WPH", 100),
        "VALUE_PER_WAFER": globals().get("VALUE_PER_WAFER", 5000),
        "REPAIR_HOURS": globals().get("REPAIR_HOURS", 8),
        "PLANNING_HORIZON": globals().get("PLANNING_HORIZON", 30),
        "BATCH_ID": globals().get("BATCH_ID", "LOT_001"),
        "total_wafers": len(binary_labels),
        "low_conf_count": int((confidences < globals().get("CONFIDENCE_THRESHOLD", 0.7)).sum()),
    }
    df_financial2, summary2 = wg_fin.compute_financials(df_batch2, config)

# Preview and save
print("--- Financial preview (top 5) ---")
print(df_financial2.head(5).to_string(index=False))
print()
print("--- Summary payload (truncated) ---")
print(json.dumps(summary2, indent=2)[:1000])
outdir_demo = os.path.join(repo_root, f"reports_{BATCH_ID}", "refactor_demo")
wg_fin.save_reports(df_batch2, df_financial2, summary2, outdir_demo, BATCH_ID)
print("Saved refactor reports →", outdir_demo)

Using repo_root: /Users/hernanmontoyag/Desktop/Capstone/waferguard-ml
models_dir= /Users/hernanmontoyag/Desktop/Capstone/waferguard-ml/wafer_images/phase2_models
No artifacts.pkl; will try to reconstruct mappings from model_testing and data: [Errno 2] No such file or directory: 'training_artifacts/artifacts.pkl'
Loading dataset from /Users/hernanmontoyag/Desktop/Capstone/waferguard-ml/data/mixedtype-wafer-defect-datasets/Wafer_Map_Datasets.npz
Loaded dataset shape: (38015, 52, 52, 3)
Model paths: /Users/hernanmontoyag/Desktop/Capstone/waferguard-ml/wafer_images/phase2_models/model_p2_cnn.keras /Users/hernanmontoyag/Desktop/Capstone/waferguard-ml/wafer_images/phase2_models/model_p2_tl.keras
--- Financial preview (top 5) ---
binary_label                pattern_name  count  batch_pct  avg_confidence  triage_priority risk_level yield_range  yield_mid  repair_cost  replacement_cost  downtime_per_hr  weighted_daily_loss  break_even_days    evoa_30d  priority_score                            